#  Importacion de librerias

In [1]:
import pandas as pd 
from pathlib import Path

print("version de pandas: ", pd.__version__)

version de pandas:  3.0.6


## Verificacion de Ruta

In [2]:
ruta = Path("../../data/bronze_simple/data/bronze_simple/cartera_diaria/cartera_diaria.csv")

ruta = Path("../../data/bronze_simple/data/bronze_simple/cartera_diaria/cartera_diaria.csv")

print("exixte el archivo? ", ruta.exists())

exixte el archivo?  True


## Lectura de Datos 

In [3]:
df_cartera = pd.read_csv(filepath_or_buffer=ruta, sep=",", encoding="UTF-8",skiprows = 1 )
df_cartera.head(5)

,credito_id,fecha_corte,dias_mora,saldo_capital,saldo_vencido,cuotas_vencidas,valor_exigible,estado_mora
0,CR000001,2024-07-01,0.0,13000000,0,0,0,AL_DIA
1,CR000001,2024-07-02,0.0,13000000,0,0,0,AL_DIA
2,CR000001,2024-07-03,0.0,13000000,0,0,0,AL_DIA
3,CR000001,2024-07-04,0.0,13000000,0,0,0,AL_DIA
4,CR000001,2024-07-05,0.0,13000000,0,0,0,AL_DIA


In [48]:
print("Dimensiones", df_cartera.shape)

Dimensiones (35695, 8)


In [49]:
print(df_cartera[["credito_id", "fecha_corte", "dias_mora", "saldo_vencido"]].isna().sum())

credito_id       0
fecha_corte      0
dias_mora        3
saldo_vencido    0
dtype: int64


In [50]:
duplicados = df_cartera.duplicated(subset=["credito_id", "fecha_corte"], keep=False)
print("Duplicados", duplicados.sum())

Duplicados 20


In [51]:
df_cartera.info()

<class 'pandas.DataFrame'>
RangeIndex: 35695 entries, 0 to 35694
Data columns (total 8 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   credito_id       35695 non-null  str    
 1   fecha_corte      35695 non-null  str    
 2   dias_mora        35692 non-null  float64
 3   saldo_capital    35695 non-null  int64  
 4   saldo_vencido    35695 non-null  int64  
 5   cuotas_vencidas  35695 non-null  int64  
 6   valor_exigible   35695 non-null  int64  
 7   estado_mora      35695 non-null  str    
dtypes: float64(1), int64(4), str(3)
memory usage: 2.2 MB


## Obervar un credito 

In [ ]:
df_cartera["fecha_corte"] = pd.to_datetime(df_cartera["fecha_corte"], errors = "coerce")

candidatos = df_cartera.loc[df_cartera["dias_mora"].between(1,30), "credito_id"
].dropna().unique()

print(" cantidad de creditos candidatos a target: ", len(candidatos))

if len(candidatos) == 0:
    raise ValueError("No existen creditos con mora entre 1 y 30 dias") 


credito_ejemplo = candidatos[0]
historial = df_cartera.loc[df_cartera["credito_id"]== credito_ejemplo].sort_values("fecha_corte")
print("credito seleccionado: ", credito_ejemplo)
display(historial[["fecha_corte", "dias_mora", "saldo_vencido"]].head(30))

 cantidad de creditos candidatos a target:  49
credito seleccionado:  CR000001


,fecha_corte,dias_mora,saldo_vencido
0,2024-07-01,0.0,0
1,2024-07-02,0.0,0
2,2024-07-03,0.0,0
3,2024-07-04,0.0,0
4,2024-07-05,0.0,0
5,2024-07-06,0.0,0
6,2024-07-07,0.0,0
7,2024-07-08,0.0,0
8,2024-07-09,0.0,0
9,2024-07-10,0.0,0


## Definicion del target
**Regla de negocio:** para cada credito que entra en mora temprana (dia `t0`), `target = 1` si en la ventana de 0 a 30 dias posteriores a `t0` el credito queda con `dias_mora = 0` **y** `saldo_vencido = 0` (se regulariza); `target = 0` en caso contrario.

## Paso 1. Limpieza

In [ ]:
# Duplicados exactos (misma fila repetida) -> se elimina una copia
n0 = len(df_cartera)
df = df_cartera.drop_duplicates().copy()
print("Filas eliminadas:", n0 - len(df))
print("Duplicados restantes por (credito, fecha):", df.duplicated(["credito_id", "fecha_corte"]).sum())

# Nulos en dias_mora: los 3 casos estan AL_DIA -> se imputa 0 (se valida antes)
nulos = df["dias_mora"].isna()
print("Nulos de dias_mora por estado:\n", df.loc[nulos, "estado_mora"].value_counts())
df.loc[nulos & (df["estado_mora"] == "AL_DIA"), "dias_mora"] = 0

df = df.sort_values(["credito_id", "fecha_corte"]).reset_index(drop=True)

## Paso 2. Parametros y evento de entrada (`t0`)
`t0` = primer dia de cada episodio de mora (dia en mora cuyo dia anterior estaba al dia). Solo se conservan entradas con mora entre 1 y 30 dias.

In [ ]:
HORIZONTE = 30
fecha_max = df["fecha_corte"].max()

df["en_mora"] = df["dias_mora"] > 0
df["en_mora_prev"] = df.groupby("credito_id")["en_mora"].shift(1, fill_value=False)

entradas = df[df["en_mora"] & ~df["en_mora_prev"] & df["dias_mora"].between(1, 30)].copy()
print("Episodios de mora temprana:", len(entradas), "| creditos:", entradas["credito_id"].nunique())

## Paso 3. Etiqueta
Para cada `t0` se miran los dias `t0+1 ... t0+30`. Si algun dia cumple `dias_mora == 0` y `saldo_vencido == 0` -> `target = 1`.

In [ ]:
df["regularizado"] = (df["dias_mora"] == 0) & (df["saldo_vencido"] == 0)
por_credito = {k: v for k, v in df.groupby("credito_id")}

def etiquetar(fila):
    v = por_credito[fila["credito_id"]]
    fin = fila["fecha_corte"] + pd.Timedelta(days=HORIZONTE)
    ventana = v[(v["fecha_corte"] > fila["fecha_corte"]) & (v["fecha_corte"] <= fin)]
    return pd.Series({
        "fecha_fin_ventana": fin,
        "ventana_completa": fin <= fecha_max,
        "target": int(ventana["regularizado"].any()),
    })

entradas = entradas.join(entradas.apply(etiquetar, axis=1))
entradas[["credito_id", "fecha_corte", "dias_mora", "fecha_fin_ventana", "ventana_completa", "target"]].head(10)

## Paso 4. Censura
Se descartan los `t0` cuya ventana de 30 dias excede el ultimo corte disponible: no se sabe si se regularizaron.

In [ ]:
print("Con ventana incompleta (descartados):", (~entradas["ventana_completa"]).sum())
df_target = entradas[entradas["ventana_completa"]].copy()
print("Filas finales:", len(df_target))

## Paso 5. Validacion

In [ ]:
print(df_target["target"].value_counts())
print(df_target["target"].value_counts(normalize=True).round(3))
print("Creditos unicos:", df_target["credito_id"].nunique())
print("Episodios por credito:\n", df_target.groupby("credito_id").size().describe())

In [ ]:
# Revision manual: un credito con target=1 y uno con target=0
for t in (1, 0):
    f = df_target[df_target["target"] == t].iloc[0]
    h = df[(df["credito_id"] == f["credito_id"]) &
           (df["fecha_corte"].between(f["fecha_corte"], f["fecha_fin_ventana"]))]
    print(f"target={t} | {f['credito_id']} | t0={f['fecha_corte'].date()}")
    display(h[["fecha_corte", "dias_mora", "saldo_vencido"]].head(32))

## Paso 6. Guardar

In [ ]:
salida = df_target[["credito_id", "fecha_corte", "dias_mora", "saldo_vencido", "fecha_fin_ventana", "target"]]
salida = salida.rename(columns={"fecha_corte": "fecha_t0"})
ruta_salida = Path("../../data/target.csv")
salida.to_csv(ruta_salida, index=False)
print("Guardado:", ruta_salida.resolve(), salida.shape)